# 02. 지자체 복지서비스 적재 (resume 내장)

> **목적**: 한국사회보장정보원 `지자체복지서비스` API에서 목록 + 상세를 받아
> 정형화 후 Supabase에 적재. 텍스트 기반 자격조건도 함께 추출.

## 두 단계 흐름
```
[GET] LcgvWelfarelist       ← 페이지네이션 ~4,500건 (기본 정보)
       │
       ▼
   raw_data UPSERT (기본 필드만)
       │
       ▼
[GET] LcgvWelfaredetailed   ← servId 단건 조회 (자격조건 텍스트)
       │                      ⚠️ 일일 한도 1,000건 → resume 자동 처리
       ▼
   raw_data 병합 + UPDATE
       │
       ▼
   정형화 + 키워드/연령 추출
       │
       ├──► welfare_services UPDATE (정형화된 필드)
       └──► welfare_support_conditions UPSERT (boolean·연령)
```

## resume 기능
- 이미 detail 받은 servId(`raw_data`에 `sprtTrgtCn` 있음)는 자동 skip
- 일일 한도 도달 시 자동 정지, 다음 날 다시 실행하면 이어서 진행
- 완료까지 약 4~5일 소요 (4,500건 ÷ 1,000건/일)

## 소요 시간
- 첫날: ~30분 (목록 5분 + 상세 1,000건 25분 + 정형화 1분)
- 이후: ~25분/일 (상세 1,000건씩)


## 1. 환경 설정

In [22]:
# !pip install supabase requests python-dotenv


In [1]:
from dotenv import load_dotenv
from pathlib import Path
import os, json, time, re
from datetime import datetime, timezone
from urllib.parse import unquote
import requests
import xml.etree.ElementTree as ET

load_dotenv(".env", override=True)

REQUIRED = ["LOCAL_WELFARE_API_KEY", "SUPABASE_URL", "SUPABASE_SERVICE_KEY"]
missing = [k for k in REQUIRED if not os.getenv(k) or str(os.getenv(k,"")).startswith(("발급","https://your"))]
if missing:
    print(f"⚠️ 필수 키 누락: {missing}")
else:
    print("✅ 필수 키 OK")

from supabase import create_client, Client
SB: Client = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_SERVICE_KEY"))
r = SB.table("welfare_services").select("id", count="exact").limit(1).execute()
print(f"✅ Supabase OK (welfare_services 총 {r.count:,}행)")


✅ 필수 키 OK
✅ Supabase OK (welfare_services 총 15,525행)


## 2. API 클라이언트

In [2]:
BASE = "https://apis.data.go.kr/B554287/LocalGovernmentWelfareInformations"

def _key():
    raw = os.getenv("LOCAL_WELFARE_API_KEY", "")
    return unquote(raw) if "%" in raw else raw

def fetch_list_pages(num_rows=100, max_pages=50, sleep=0.3):
    """LcgvWelfarelist - 전체 페이지 (목록만)"""
    all_items = []
    for page in range(1, max_pages + 1):
        params = {"serviceKey": _key(), "pageNo": page, "numOfRows": num_rows}
        r = requests.get(f"{BASE}/LcgvWelfarelist", params=params, timeout=30)
        r.raise_for_status()
        try:
            root = ET.fromstring(r.text)
            items = []
            for node in root.findall(".//servList") + root.findall(".//item"):
                items.append({c.tag: (c.text or "").strip() for c in node})
        except ET.ParseError:
            print(f"  ⚠️ page {page} XML 파싱 실패")
            break
        all_items.extend(items)
        print(f"  page {page:2d}: +{len(items)} (누적 {len(all_items):,})")
        if len(items) < num_rows: break
        time.sleep(sleep)
    return all_items

def parse_detail_xml(xml_text: str) -> dict:
    """LcgvWelfaredetailed XML → dict (wantedDtl 자식들)"""
    try:
        root = ET.fromstring(xml_text)
    except ET.ParseError:
        return {}
    # 응답 root는 <wantedDtl>이고 자식들이 데이터 노드
    if root.tag == "wantedDtl" or root.find("wantedDtl") is not None:
        target = root if root.tag == "wantedDtl" else root.find("wantedDtl")
        return {c.tag: (c.text or "").strip() for c in target}
    # fallback
    for tag in ["servDetail","servList","item"]:
        nodes = root.findall(f".//{tag}")
        if nodes:
            return {c.tag: (c.text or "").strip() for c in nodes[0]}
    return {c.tag: (c.text or "").strip() for c in root if c.tag not in ("header","body","resultCode","resultMessage")}

def fetch_detail(servId: str, sess=None, retries=2):
    sess = sess or requests
    params = {"serviceKey": _key(), "servId": servId}
    for attempt in range(retries + 1):
        try:
            r = sess.get(f"{BASE}/LcgvWelfaredetailed", params=params, timeout=15)
            if r.status_code == 200 and r.text.strip():
                d = parse_detail_xml(r.text)
                # 응답 성공 여부 확인 - sprtTrgtCn 등 핵심 키 존재해야 진짜 성공
                if d and (d.get("servId") or d.get("sprtTrgtCn") or d.get("servNm")):
                    return d
        except Exception:
            pass
        if attempt < retries: time.sleep(0.5)
    return None

print("API 함수 등록")


API 함수 등록


## 3. 정형화 함수 (오타 패치 포함)

In [3]:
def _s(v):
    if v is None: return None
    s = str(v).strip()
    return s if s else None

def _first(d: dict, keys: list):
    for k in keys:
        v = d.get(k)
        if v not in (None, "", []): return v
    return None

def _split_arr(v):
    if v is None: return None
    if isinstance(v, list): return [str(x).strip() for x in v if str(x).strip()]
    s = str(v).strip()
    if not s: return None
    for sep in [",","|",";","/"]:
        if sep in s:
            return [t.strip() for t in s.split(sep) if t.strip()]
    return [s]

def _apply_deadline(m: dict):
    se = str(m.get("aplyPrdSeCd","")).strip()
    if se in ("1","상시"): return "상시"
    bgng = m.get("bgngYmd") or m.get("enfcBgngYmd")
    end  = m.get("endYmd")  or m.get("enfcEndYmd")
    if bgng and end and end != "99991231": return f"{bgng} ~ {end}"
    if bgng: return f"{bgng}~"
    return None

def normalize_local(raw: dict, sid: str) -> dict:
    """raw_data(목록+상세 병합) → welfare_services 행"""
    sido = _s(_first(raw, ["ctpvNm"]))
    sigungu = _s(_first(raw, ["sggNm"]))
    agency = " ".join([p for p in [sido, sigungu] if p]) or _s(raw.get("jurMnofNm"))
    return {
        "source": "지자체",
        "service_id": sid,
        "service_name":       _s(_first(raw, ["servNm"])),
        "service_summary":    _s(_first(raw, ["servDgst"])),
        "agency_name":        agency,
        "agency_type":        "지자체",
        "department":         _s(_first(raw, ["bizChrDeptNm"])),
        "support_type":       _s(_first(raw, ["srvPvsnNm","sprtCycNm"])),
        "user_type":          _s(_first(raw, ["trgterIndvdlNmArray","trgterIndvdlArray"])),
        "service_field":      _s(_first(raw, ["intrsThemaNmArray"])),
        # 패치된 키 (실제 응답 키)
        "target_description": _s(_first(raw, ["sprtTrgtCn"])),
        "selection_criteria": _s(_first(raw, ["slctCritCn"])),
        "support_content":    _s(_first(raw, ["alwServCn","servDgst"])),
        "apply_method":       _s(_first(raw, ["aplyMtdCn"])),
        "apply_deadline":     _apply_deadline(raw),
        "receiving_agency":   _s(_first(raw, ["rcptInsttNm"])),
        "contact":            _s(_first(raw, ["inqplCtadrCn","rprsCtadr"])),
        "detail_url":         _s(_first(raw, ["servDtlLink"])),
        "region_sido":        sido,
        "region_sigungu":     sigungu,
        "life_stages":        _split_arr(_first(raw, ["lifeNmArray","lifeArray"])),
        "interest_themes":    _split_arr(_first(raw, ["intrsThemaNmArray"])),
        "raw_data":           raw,
    }

print("정형화 함수 등록")


정형화 함수 등록


## 4. 자격조건 추출기 (키워드/정규식)

In [4]:
AGE_PATTERNS = [
    r"만\s*(\d{1,2})\s*세\s*이상",
    r"(\d{1,2})\s*세\s*이상",
    r"만\s*(\d{1,2})\s*세\s*이하",
    r"(\d{1,2})\s*세\s*이하",
    r"(\d{1,2})\s*~\s*(\d{1,2})\s*세",
]

KEYWORD_TO_FLAG = {
    "disabled":           ["장애인","중증장애","지체장애","발달장애"],
    "veteran":            ["국가보훈","보훈대상","유공자"],
    "illness":            ["희귀질환","중증질환","만성질환","암환자"],
    "single_parent":      ["한부모","조손"],
    "single_household":   ["1인가구","독거"],
    "multi_child":        ["다자녀","세 자녀"],
    "no_house":           ["무주택"],
    "expecting_parent":   ["예비부모","난임"],
    "pregnant":           ["임산부","임신","임부"],
    "postpartum":         ["출산","산모","산후","입양"],
    "farmer":             ["농업인","농민","농가"],
    "fisher":             ["어업인","어민","어가"],
    "livestock":          ["축산업","축산농가"],
    "forester":           ["임업인"],
    "elementary":         ["초등학생"],
    "middle_school":      ["중학생"],
    "high_school":        ["고등학생"],
    "university":         ["대학생","대학원생"],
    "employed":           ["근로자","직장인","재직자"],
    "unemployed":         ["구직자","실업자","미취업"],
    "multi_cultural":     ["다문화"],
    "north_korean_defector": ["북한이탈"],
}

INCOME_KEYWORDS = {
    "income_band_50":  ["중위소득 50%","기준중위소득 50","기초생활수급"],
    "income_band_75":  ["중위소득 75%","차상위"],
    "income_band_100": ["중위소득 100%"],
    "income_band_200": ["중위소득 200%"],
}

def _ages(text: str):
    if not text: return None, None
    age_s, age_e = None, None
    for pat in AGE_PATTERNS:
        for m in re.finditer(pat, text):
            g = m.groups()
            if "이상" in m.group(0): age_s = max(age_s or 0, int(g[0]))
            elif "이하" in m.group(0): age_e = min(age_e or 200, int(g[0]))
            elif "~" in m.group(0) and len(g) >= 2:
                age_s = max(age_s or 0, int(g[0]))
                age_e = min(age_e or 200, int(g[1]))
    return age_s, age_e

def extract_conditions(raw: dict, sid: str) -> dict:
    text_keys = ["sprtTrgtCn","slctCritCn","alwServCn",
                 "trgterIndvdlNmArray","intrsThemaNmArray","lifeNmArray","servDgst"]
    text = " ".join(str(raw.get(k)) for k in text_keys if raw.get(k))
    age_s, age_e = _ages(text)
    cond = {
        "service_id": sid,
        "age_start": age_s, "age_end": age_e,
        "male_eligible": True, "female_eligible": True,
        "raw_codes": {"_from": "지자체_텍스트", "_text_len": len(text)},
        "fetched_at": datetime.now(timezone.utc).isoformat(),
    }
    for f in list(KEYWORD_TO_FLAG.keys()) + list(INCOME_KEYWORDS.keys()):
        cond[f] = False
    for f, kws in KEYWORD_TO_FLAG.items():
        if any(k in text for k in kws): cond[f] = True
    for f, kws in INCOME_KEYWORDS.items():
        if any(k in text for k in kws): cond[f] = True
    return cond

print("자격조건 추출기 등록")


자격조건 추출기 등록


## 5. 목록 적재 (LcgvWelfarelist) — 항상 갱신

In [5]:
def chunked(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i+size]

def upsert_table(table, rows, chunk=200):
    if not rows: return 0
    total = 0
    for batch in chunked(rows, chunk):
        SB.table(table).upsert(batch, on_conflict="service_id").execute()
        total += len(batch)
    return total

print("="*60); print("[1/3] 지자체 복지 목록 fetch"); print("="*60)
raw_list = fetch_list_pages(num_rows=100, max_pages=50)
print(f"\n수집: {len(raw_list):,}건")

# 중복 제거 후 servId 기준 목록만 먼저 적재 (raw_data는 목록 응답 그대로)
seen, dedup = set(), []
for r in raw_list:
    sid = r.get("servId") or r.get("서비스ID")
    if sid and sid not in seen:
        seen.add(sid); dedup.append({"servId": sid, "raw": r})
print(f"중복 제거: {len(raw_list):,} → {len(dedup):,}")

# 기존 raw_data와 병합 (이미 detail이 있으면 보존하기 위해)
existing = {}
offset = 0
while True:
    r = (SB.table("welfare_services")
         .select("service_id, raw_data").eq("source","지자체")
         .range(offset, offset+999).execute())
    if not r.data: break
    for row in r.data:
        existing[row["service_id"]] = row.get("raw_data") or {}
    if len(r.data) < 1000: break
    offset += 1000
print(f"기존 raw_data 로드: {len(existing):,}개")

# 정형화 + UPSERT (raw_data는 기존+목록 병합)
norm_rows = []
for item in dedup:
    sid = item["servId"]
    merged_raw = {**existing.get(sid, {}), **item["raw"]}
    norm_rows.append(normalize_local(merged_raw, sid))

print(f"\nwelfare_services upsert ({len(norm_rows):,}건)...")
upserted = upsert_table("welfare_services", norm_rows, chunk=200)
print(f"✅ {upserted:,}건 갱신")


[1/3] 지자체 복지 목록 fetch
  page  1: +100 (누적 100)
  page  2: +100 (누적 200)
  page  3: +100 (누적 300)
  page  4: +100 (누적 400)
  page  5: +100 (누적 500)
  page  6: +100 (누적 600)
  page  7: +100 (누적 700)
  page  8: +100 (누적 800)
  page  9: +100 (누적 900)
  page 10: +100 (누적 1,000)
  page 11: +100 (누적 1,100)
  page 12: +100 (누적 1,200)
  page 13: +100 (누적 1,300)
  page 14: +100 (누적 1,400)
  page 15: +100 (누적 1,500)
  page 16: +100 (누적 1,600)
  page 17: +100 (누적 1,700)
  page 18: +100 (누적 1,800)
  page 19: +100 (누적 1,900)
  page 20: +100 (누적 2,000)
  page 21: +100 (누적 2,100)
  page 22: +100 (누적 2,200)
  page 23: +100 (누적 2,300)
  page 24: +100 (누적 2,400)
  page 25: +100 (누적 2,500)
  page 26: +100 (누적 2,600)
  page 27: +100 (누적 2,700)
  page 28: +100 (누적 2,800)
  page 29: +100 (누적 2,900)
  page 30: +100 (누적 3,000)
  page 31: +100 (누적 3,100)
  page 32: +100 (누적 3,200)
  page 33: +100 (누적 3,300)
  page 34: +100 (누적 3,400)
  page 35: +100 (누적 3,500)
  page 36: +100 (누적 3,600)
  page 37: +100 (누적 3,70

## 6. 상세 적재 (LcgvWelfaredetailed) — resume 내장

`raw_data`에 `sprtTrgtCn`이 이미 있는 servId는 자동 skip.
일일 한도 1,000건. 한도 도달 시 자동 정지 — 다음 날 다시 실행.


In [6]:
# 모든 지자체 servId 다시 fetch (detail 있는지 판단)
all_rows = []
offset = 0
while True:
    r = (SB.table("welfare_services")
         .select("service_id, raw_data").eq("source","지자체")
         .range(offset, offset+999).execute())
    if not r.data: break
    all_rows.extend(r.data)
    if len(r.data) < 1000: break
    offset += 1000

# detail 대상 = sprtTrgtCn 없는 servId
pending = [row for row in all_rows if not (row.get("raw_data") or {}).get("sprtTrgtCn")]
done = len(all_rows) - len(pending)
print(f"전체 {len(all_rows):,}건  |  detail 완료 {done:,}건  |  남은 {len(pending):,}건")

DAILY_LIMIT = 990   # 안전 마진 (실제 limit 1,000)
target = pending[:DAILY_LIMIT]
print(f"\n오늘 호출 대상: {len(target):,}건 (일일 한도 {DAILY_LIMIT}건)\n")


전체 4,561건  |  detail 완료 2,999건  |  남은 1,562건

오늘 호출 대상: 990건 (일일 한도 990건)



In [7]:
import threading
sess = requests.Session()

details_map = {}
errors = 0
START = time.time()
STOP_ERRORS_IN_A_ROW = 50  # 연속 50건 실패 → 한도 도달로 보고 정지
errors_in_a_row = 0

for i, row in enumerate(target, 1):
    sid = row["service_id"]
    d = fetch_detail(sid, sess=sess)
    if d:
        details_map[sid] = d
        errors_in_a_row = 0
    else:
        errors += 1
        errors_in_a_row += 1
    if i % 100 == 0 or i == len(target):
        elapsed = time.time() - START
        rem = elapsed / i * (len(target) - i)
        print(f"  [{i:>4}/{len(target):,}] 성공 {len(details_map):,} | 에러 {errors} | 경과 {elapsed/60:.1f}m | 남음 {rem/60:.1f}m")
    if errors_in_a_row >= STOP_ERRORS_IN_A_ROW:
        print(f"\n⚠️ 연속 {STOP_ERRORS_IN_A_ROW}건 실패 — 일일 한도 도달 추정. 정지합니다.")
        break
    time.sleep(0.12)

print(f"\n✅ detail 수집: {len(details_map):,}건 / 에러 {errors} / {(time.time()-START)/60:.1f}분")
print(f"   남은 {len(pending) - len(details_map):,}건은 내일 다시 실행하면 이어서 진행됩니다.")



⚠️ 연속 50건 실패 — 일일 한도 도달 추정. 정지합니다.

✅ detail 수집: 0건 / 에러 50 / 1.0분
   남은 1,562건은 내일 다시 실행하면 이어서 진행됩니다.


## 7. 정형화 + UPSERT (방금 받은 detail로 갱신)

In [8]:
# 갱신할 행: 이번에 받은 detail 있는 것만
updated_rows = []
cond_rows = []
for row in all_rows:
    sid = row["service_id"]
    base_raw = row.get("raw_data") or {}
    detail = details_map.get(sid)
    if detail:
        merged = {**base_raw, **detail}
    elif base_raw.get("sprtTrgtCn"):
        merged = base_raw  # 이미 이전에 detail 받은 행 → 그대로 재정형화
    else:
        continue  # 아직 detail 없음 → 건너뜀
    updated_rows.append(normalize_local(merged, sid))
    cond_rows.append(extract_conditions(merged, sid))

print(f"정형화: {len(updated_rows):,}건 갱신 예정")

# 채워진 비율
filled = {k: sum(1 for s in updated_rows if s[k]) for k in
          ["target_description","selection_criteria","support_content","service_field"]}
print("\n📊 새 채워진 비율 (detail 완료된 행만):")
for k, c in filled.items():
    print(f"  {k:22s} {c:>5,}/{len(updated_rows):,}  ({100*c/max(1,len(updated_rows)):5.1f}%)")


정형화: 2,999건 갱신 예정

📊 새 채워진 비율 (detail 완료된 행만):
  target_description     2,999/2,999  (100.0%)
  selection_criteria     2,999/2,999  (100.0%)
  support_content        2,999/2,999  (100.0%)
  service_field          2,251/2,999  ( 75.1%)


In [9]:
print("\nwelfare_services UPDATE...")
u1 = upsert_table("welfare_services", updated_rows, chunk=200)
print(f"✅ {u1:,}건 갱신")

print("\nwelfare_support_conditions UPSERT...")
u2 = upsert_table("welfare_support_conditions", cond_rows, chunk=200)
print(f"✅ {u2:,}건 갱신")



welfare_services UPDATE...
✅ 2,999건 갱신

welfare_support_conditions UPSERT...
✅ 2,999건 갱신


## 8. 검증

In [10]:
r_svc = SB.table("welfare_services").select("id", count="exact").limit(1).execute()
r_loc = SB.table("welfare_services").select("id", count="exact").eq("source","지자체").limit(1).execute()
r_cond = SB.table("welfare_support_conditions").select("id", count="exact").limit(1).execute()
print("="*50)
print(f"welfare_services         : {r_svc.count:,}")
print(f"  └ 지자체               : {r_loc.count:,}")
print(f"welfare_support_conditions: {r_cond.count:,}")
print("="*50)

sample = (SB.table("welfare_services")
          .select("target_description, selection_criteria, support_content, service_field")
          .eq("source","지자체").limit(1000).execute().data)
print("\n지자체 1,000건 샘플 NULL 채워진 비율:")
for k in ["target_description","selection_criteria","support_content","service_field"]:
    c = sum(1 for s in sample if s.get(k))
    print(f"  {k:22s} {c:>4,}/{len(sample):,}  ({100*c/len(sample):.1f}%)")


welfare_services         : 15,525
  └ 지자체               : 4,561
welfare_support_conditions: 15,514

지자체 1,000건 샘플 NULL 채워진 비율:
  target_description     1,000/1,000  (100.0%)
  selection_criteria     1,000/1,000  (100.0%)
  support_content        1,000/1,000  (100.0%)
  service_field           755/1,000  (75.5%)


## 다음 단계

### 완료된 경우
→ **03_welfare_agent.ipynb** 실행 (시연)

### 일일 한도로 중단된 경우
→ 24시간 뒤 이 노트북 **Run All** 한 번 더. resume이 자동으로 동작합니다.
→ 4~5일 후 모든 detail 완료.
